# Chelsea-style holdout test on Dallas

The Chelsea study trained on properties in a proactively inspected target area and
scored the rest of the city, where its test labels existed only for properties that
happened to be inspected. Dallas inspects every multitenant property, so the same
design can be run with the outcome known everywhere: treat one part of the city as
the only inspected area, train there, and score the remainder.

**Question:** would a Chelsea-style model have found Dallas's low-scoring properties
better than random inspection, or better than a one-variable rule (rank by tract
poverty)?

**Design**

- **Chelsea mirror (headline).** Train on properties in the highest-poverty third of
  tracts; test on everything else. Chelsea's target area was also its densest, poorest
  area, so this keeps the shift Chelsea had.
- **Random 70/30 split.** Reference ceiling, no shift.
- **Grouped 5-fold CV by tract.** Generalization to unseen tracts without the poverty
  shift.

**Targets**

- `bottom10`: ever scored in the bottom 10% (score ≤ 86) across FY2020-21 to FY2024-25.
- `bottom10_first`: bottom 10% on the property's first scored inspection. Re-inspection
  is risk-based (1-2-3 year cadence), so "ever" partly measures how often a property was
  seen. The first inspection puts every property on one inspection.

Bottom 10% is a statistical cut. Code Compliance treats 70 and above as passing.

**Two Chelsea code issues not carried over:** it re-fit the scaler on the test set, and
it scored average precision on hard 0/1 predictions rather than probabilities. Here the
scaler is fit on train only and every metric uses predicted probabilities.

## 1. Data

All inputs are copied into the gitignored `_data/` folder; nothing here is committed.

| File | Source | Used for |
|---|---|---|
| `multitenant_properties.dta` | shared drive `_Multitenant_Inspection/_data/_analytic/` (July 2026 rebuild) | base table, targets, DCAD values, owner fields |
| `multitenant_inspections.dta` | same folder | building and pool counts, first-inspection target |
| `properties_with_acs.csv` | shared drive `_Multitenant_Inspection/_output/` | tract ACS columns only (its inspection-derived columns predate the rebuild) |
| `parcel_match.csv` | shared drive `_Multitenant_Inspection/_data/_processed/` | DCAD account key |
| `dcad_improvement_commercial.parquet` | `silver.silver_tx_dallas_cad_car.improvement_commercial`, appraisal years 2021 and 2026 | year built, condition, quality, depreciation |

To rebuild the DCAD pull:

```bash
pier query "SELECT universal_id, appraisal_year, bldg_class_desc, year_built, remodel_year,
  property_quality, property_condition, phys_depr_pct, total_depr_pct, gross_bldg_area,
  num_stories, num_units
  FROM silver.silver_tx_dallas_cad_car.improvement_commercial
  WHERE appraisal_year IN (2021, 2026)" -o _data/dcad_improvement_commercial.parquet
```

In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import GridSearchCV, GroupKFold, StratifiedKFold, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

DATA = Path("../_data")
LOG = Path("../_output/progress.log")
SEED = 1
N_JOBS = 6  # more workers than this pushes an 8 GB machine into swap
pd.set_option("display.float_format", "{:.3f}".format)

In [ ]:
ACS_COLS = [
    "pct_poverty", "pct_renter", "pct_hispanic", "pct_nh_black", "pct_nh_white",
    "pct_nh_asian", "pct_foreign_born", "pct_rent_burden_30", "pct_rent_burden_35",
]
RACE_COLS = ["pct_hispanic", "pct_nh_black", "pct_nh_white", "pct_nh_asian"]

props = pd.read_stata(DATA / "multitenant_properties.dta", convert_categoricals=False)
insp = pd.read_stata(DATA / "multitenant_inspections.dta", convert_categoricals=False)
acs = pd.read_csv(
    DATA / "properties_with_acs.csv",
    usecols=["address", "tract_geoid", *ACS_COLS],
    dtype={"tract_geoid": str},
)
parcel = pd.read_csv(DATA / "parcel_match.csv", usecols=["address", "universal_id"], dtype=str)
dcad = pd.read_parquet(DATA / "dcad_improvement_commercial.parquet")

print("properties:", props.shape, "| inspections:", insp.shape)
print("ever bottom10:", int(props.bottom10.sum()), f"({props.bottom10.mean():.1%})")
print("chronic:", int(props.chronic.sum()))

### DCAD commercial improvements → one row per account

Use the 2021 appraisal (before most inspections) when the account has one, else 2026.
Multi-building accounts: earliest year built, total area, tallest building, and
area-weighted condition, quality and physical depreciation.

In [ ]:
COND = {"POOR": 1, "FAIR": 2, "AVERAGE": 3, "GOOD": 4, "EXCELLENT": 5}
QUAL = {"LOW COST": 1, "AVERAGE": 2, "GOOD": 3, "VERY GOOD": 4, "EXCELLENT": 5}

has_2021 = set(dcad.loc[dcad.appraisal_year == 2021, "universal_id"])
imp = dcad[(dcad.appraisal_year == 2021) | ~dcad.universal_id.isin(has_2021)]
imp = imp[imp.universal_id.isin(set(parcel.universal_id.dropna()))].assign(
    cond=lambda d: d.property_condition.map(COND),
    qual=lambda d: d.property_quality.map(QUAL),
    year_built=lambda d: d.year_built.where(d.year_built.between(1850, 2026)),
    area=lambda d: d.gross_bldg_area.clip(lower=0),
)
imp["w"] = imp.area.where(imp.area > 0, 1)
for c in ["cond", "qual", "phys_depr_pct"]:
    imp[f"{c}_wx"] = imp[c] * imp.w
    imp[f"{c}_w"] = imp.w.where(imp[c].notna())

g = imp.groupby("universal_id")
dcad_acct = pd.DataFrame({
    "year_built": g.year_built.min(),
    "bldg_area": g.area.sum(),
    "n_stories": g.num_stories.max(),
    **{
        f"dcad_{c}": g[f"{c}_wx"].sum(min_count=1) / g[f"{c}_w"].sum(min_count=1)
        for c in ["cond", "qual", "phys_depr_pct"]
    },
}).reset_index()
print("DCAD accounts with improvement detail:", len(dcad_acct))

In [ ]:
insp = insp[insp.address.str.strip() != ""]
bldg = insp.groupby("address").agg(n_buildings=("n_buildings", "max"), n_pools=("n_pools", "max"))
first = (
    insp.dropna(subset=["score"])
    .sort_values(["address", "date_opened", "sr_number"])
    .groupby("address")
    .head(1)
    .set_index("address")["bottom10"]
    .rename("bottom10_first")
)

base = props[props.address.str.strip() != ""].copy()
for c in ["owner_state", "sptd_code"]:
    base[c] = base[c].replace({"NA": np.nan, "": np.nan})

df = (
    base.merge(acs, on="address", how="left")
    .merge(parcel, on="address", how="left")
    .merge(dcad_acct, on="universal_id", how="left")
    .merge(bldg, on="address", how="left")
    .merge(first, on="address", how="left")
)
print("dropped blank-address rows:", len(props) - len(base))
print("joined to ACS:", df.pct_poverty.notna().sum(), "of", len(df))
print("DCAD account key:", df.universal_id.notna().sum(), "| DCAD improvement detail:", df.year_built.notna().sum())
print("first-inspection target available:", df.bottom10_first.notna().sum())

df = df.dropna(subset=["pct_poverty"]).reset_index(drop=True)
print("analysis rows:", len(df))

## 2. Features

Everything here is known without inspecting the property. Excluded as leakage: every
score- or finding-derived column (the data dictionary's LEAK list, the tier flags, the
`*_current` flags), owner columns built from the target (`n_props_bottom10`,
`n_props_chronic`, `flag_owner_bottom10`, `flag_owner_priority`), and `flag_owner_pe`,
which the pipeline sets by name for two owners identified from the distress results.
Latitude and longitude are dropped because trees can't extrapolate coordinates into an
unseen area.

In [ ]:
units = df.n_units.where(df.n_units > 0)
mkt = df.total_market_value.where(df.total_market_value > 0)
df = df.assign(
    log_units=np.log1p(units),
    units_per_building=units / df.n_buildings.where(df.n_buildings > 0),
    has_pool=(df.n_pools > 0).astype(float).where(df.n_pools.notna()),
    log_mkt_value=np.log(mkt),
    log_value_per_unit=np.log(mkt / units),
    imp_share=df.improvement_value / mkt,
    sptd_multifamily=(df.sptd_code == "B11").astype(float).where(df.sptd_code.notna()),
    out_of_state_owner=(df.owner_state != "TEXAS").astype(float).where(df.owner_state.notna()),
    log_owner_portfolio=np.log(df.n_props_total.where(df.n_props_total > 0)),
    building_age=2021 - df.year_built,
    log_bldg_area=np.log(df.bldg_area.where(df.bldg_area > 0)),
)

PROPERTY = [
    "log_units", "units_per_building", "has_pool", "log_mkt_value", "log_value_per_unit",
    "imp_share", "sptd_multifamily", "out_of_state_owner", "log_owner_portfolio",
    "building_age", "log_bldg_area", "n_stories", "dcad_cond", "dcad_qual", "dcad_phys_depr_pct",
]
FEATURES = ACS_COLS + PROPERTY
print(df[FEATURES].notna().mean().rename("share non-missing").to_frame().T.to_string())

## 3. Splits, models, metrics

In [ ]:
poverty_cut = df.pct_poverty.quantile(2 / 3)
in_target_area = (df.pct_poverty >= poverty_cut).to_numpy()


def splits(d: pd.DataFrame, y: np.ndarray) -> dict[str, list[tuple[np.ndarray, np.ndarray]]]:
    idx = np.arange(len(d))
    area = d.pct_poverty.to_numpy() >= poverty_cut
    tr, te = train_test_split(idx, test_size=0.3, stratify=y, random_state=SEED)
    groups = d.tract_geoid.fillna("missing").to_numpy()
    return {
        "Chelsea mirror": [(idx[area], idx[~area])],
        "Random 70/30": [(tr, te)],
        "Grouped CV by tract": list(GroupKFold(n_splits=5).split(idx, y, groups)),
    }


def pipe(est, scale: bool = False):
    steps = [SimpleImputer(strategy="median", add_indicator=True)]
    if scale:
        steps.append(StandardScaler())
    return make_pipeline(*steps, est)


MODELS = {
    "LASSO": (
        pipe(LogisticRegression(l1_ratio=1.0, solver="liblinear", max_iter=5000, random_state=SEED), scale=True),
        {"logisticregression__C": [0.01, 0.03, 0.1, 0.3, 1.0]},
    ),
    "Random forest": (
        pipe(RandomForestClassifier(n_estimators=200, max_features="sqrt", random_state=SEED, n_jobs=1)),
        {
            "randomforestclassifier__max_depth": [6, None],
            "randomforestclassifier__min_samples_leaf": [5, 20],
        },
    ),
    "XGBoost": (
        pipe(XGBClassifier(
            n_estimators=300, subsample=0.8, colsample_bytree=0.8,
            random_state=SEED, n_jobs=1, eval_metric="logloss",
        )),
        {
            "xgbclassifier__max_depth": [2, 4],
            "xgbclassifier__learning_rate": [0.05],
            "xgbclassifier__min_child_weight": [1, 5],
        },
    ),
}


def fit(name: str, X: pd.DataFrame, y: np.ndarray):
    est, grid = MODELS[name]
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    gs = GridSearchCV(clone(est), grid, scoring="average_precision", cv=cv, n_jobs=N_JOBS)
    return gs.fit(X, y).best_estimator_


def found_at(y: np.ndarray, s: np.ndarray, k: float) -> float:
    # share of all positives that fall in the top k of the ranking
    rng = np.random.default_rng(SEED)
    order = np.lexsort((rng.random(len(s)), -s))
    return y[order[: int(np.ceil(k * len(y)))]].sum() / y.sum()


def scores(y: np.ndarray, s: np.ndarray) -> dict[str, float]:
    return {
        "PR-AUC": average_precision_score(y, s),
        "ROC-AUC": roc_auc_score(y, s),
        **{f"found@{int(k * 100)}%": found_at(y, s, k) for k in (0.1, 0.2, 0.3)},
    }

In [ ]:
def run(target: str, features: list[str], models=MODELS, which=None) -> pd.DataFrame:
    d = df.dropna(subset=[target]).reset_index(drop=True)
    y = d[target].astype(int).to_numpy()
    X = d[features]
    rows = []
    for split_name, folds in splits(d, y).items():
        if which and split_name not in which:
            continue
        test = np.concatenate([te for _, te in folds])
        yt = y[test]
        pov = d.pct_poverty.to_numpy()[test]
        base = {"target": target, "split": split_name, "n_train": len(folds[0][0]),
                "n_test": len(test), "prevalence_test": yt.mean()}
        rows.append({**base, "ranker": "Random", "PR-AUC": yt.mean(), "ROC-AUC": 0.5,
                     "found@10%": 0.1, "found@20%": 0.2, "found@30%": 0.3})
        rows.append({**base, "ranker": "Poverty only", **scores(yt, pov)})
        for m in models:
            oof = np.empty(len(d))
            for tr, te in folds:
                oof[te] = fit(m, X.iloc[tr], y[tr]).predict_proba(X.iloc[te])[:, 1]
            rows.append({**base, "ranker": m, **scores(yt, oof[test])})
            with LOG.open("a") as f:
                f.write(f"{target} | {split_name} | {m} | {len(features)} features | PR-AUC {rows[-1]['PR-AUC']:.3f}\n")
    return pd.DataFrame(rows)


TARGETS = ["bottom10", "bottom10_first"]
for t in TARGETS:
    d = df.dropna(subset=[t])
    a = d.pct_poverty >= poverty_cut
    print(f"{t}: n={len(d)}, target-area n={a.sum()} prev={d.loc[a, t].mean():.3f}, "
          f"rest n={(~a).sum()} prev={d.loc[~a, t].mean():.3f}")

## 4. Results

In [ ]:
results = pd.concat([run(t, FEATURES) for t in TARGETS], ignore_index=True)
cols = ["target", "split", "ranker", "n_train", "n_test", "prevalence_test",
        "PR-AUC", "ROC-AUC", "found@10%", "found@20%", "found@30%"]
print(results[cols].to_string(index=False))

### Feature-set variants (Chelsea mirror, `bottom10`)

How much of the signal is neighborhood versus building, and how much depends on tract
race and ethnicity shares.

In [ ]:
VARIANTS = {
    "All features": FEATURES,
    "No race/ethnicity": [c for c in FEATURES if c not in RACE_COLS],
    "Property only": PROPERTY,
    "Neighborhood only": ACS_COLS,
}
variants = pd.concat(
    [run("bottom10", f, which={"Chelsea mirror"}).assign(features=name) for name, f in VARIANTS.items()],
    ignore_index=True,
)
variants = variants[~variants.ranker.isin(["Random", "Poverty only"])]
print(variants[["features", "ranker", "PR-AUC", "ROC-AUC", "found@20%"]].to_string(index=False))

## 5. Sanity checks

In [ ]:
d = df.dropna(subset=["bottom10"]).reset_index(drop=True)
y = d.bottom10.astype(int).to_numpy()
area = d.pct_poverty.to_numpy() >= poverty_cut
rng = np.random.default_rng(SEED)

shuffled = rng.permutation(y)
m = fit("Random forest", d.loc[area, FEATURES], shuffled[area])
p = m.predict_proba(d.loc[~area, FEATURES])[:, 1]
print(f"shuffled labels: PR-AUC {average_precision_score(shuffled[~area], p):.3f} "
      f"vs prevalence {shuffled[~area].mean():.3f}")

leak = FEATURES + ["score_min"]
m = fit("Random forest", d.loc[area, leak], y[area])
p = m.predict_proba(d.loc[~area, leak])[:, 1]
print(f"with score_min (leak): PR-AUC {average_precision_score(y[~area], p):.3f}")

## 6. What drives the ranking (permutation importance on the test set)

In [ ]:
def importance(split_name: str, model: str, target: str = "bottom10") -> pd.Series:
    d = df.dropna(subset=[target]).reset_index(drop=True)
    y = d[target].astype(int).to_numpy()
    (tr, te), = splits(d, y)[split_name]
    est = fit(model, d.loc[tr, FEATURES], y[tr])
    r = permutation_importance(
        est, d.loc[te, FEATURES], y[te], scoring="average_precision",
        n_repeats=20, random_state=SEED, n_jobs=N_JOBS,
    )
    return pd.Series(r.importances_mean, index=FEATURES).sort_values(ascending=False)


best = (
    results[(results.target == "bottom10") & results.ranker.isin(MODELS)]
    .groupby("ranker")["PR-AUC"].mean().idxmax()
)
print("model:", best)
imp_tbl = pd.DataFrame({s: importance(s, best) for s in ["Chelsea mirror", "Random 70/30"]})
print(imp_tbl.sort_values("Random 70/30", ascending=False).round(4).to_string())

## 7. Reading the results

- **PR-AUC** is the headline metric, as in Chelsea. Random scores the test-set prevalence.
- **found@20%** is the operational number: the share of all bottom-10 properties in the
  test area that an inspector would reach by visiting the top-ranked 20% first. Random
  finds 20%.
- **Poverty only** ranks by tract poverty rate. A model that can't beat it adds little
  beyond a map.

This model ranks property and neighborhood profiles associated with low scores. Dallas
already inspects every multitenant property, so the practical use is ordering or
spacing inspections, not deciding which properties get one.